In [ ]:
# Base class
from src.Base.Config import Config

# Project class
from src.DopaMouv.Trajectory import TrajectoryDLC, DLCTracking
from src.DopaMouv.Leds import Leds
from src.DopaMouv.Video import Video
import src.utils as u

# other lib
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

In [3]:
cfg = Config(Path("./config.yaml"))
project = cfg.load_project()

custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme("talk", style="ticks", rc=custom_params, palette="pastel")

analysis_dir = u.make_dir(project.paths.analysis(project.trialgroup.keep_val))
df_dir = u.make_dir(analysis_dir / "trials_df")

project.trialgroup.crop_laser_period = True
project.trialgroup.remove_NOstim = True

log: config_20261113_100902.yaml
========== Project : DopaMouv

 Chrimson_NoGo_Cuedriven_4p_Rat226


### LED verif

In [ ]:
from src.DopaMouv.Video import Video
from src.DopaMouv.Trial import Trial
from src.DopaMouv.Leds import Leds

dir_names = []

for clip_path in (project.paths.raw).rglob("*.mp4"): 
    clip = Video(clip_path)
    parent_dir = clip_path.parent.stem

    if parent_dir in dir_names: 
        continue

    dir_names.append(parent_dir)
    # print(f"\nextracting {clip_path} \nto {out_vid_dir / parent_dir}")
    # clip.extract_frames([0], out_vid_dir / parent_dir, clip_path)

    trial = Trial(clip_path)

    if trial.file.date.month >= 10: 
        continue

    print()

    rule_key = {
        "subject": trial.subject,
        "view": trial.camera_view,
        "month": trial.file.date.month,
    }
    annotation = u.match_rule(rule_key, project.annotation_rules)
    print(rule_key, " -> ", annotation)

    Leds(video_path=clip_path, label_studio_annotation=annotation, 
                fig_output_path=out_vid_dir / f'{parent_dir}_lum.html', 
                csv_output_path=out_vid_dir / f'{parent_dir}_lum.csv')
    


## Sunburst

In [ ]:
project.trialgroup.load_trials()
project.trialgroup.sunburst_metadata(output_dir=u.make_dir(analysis_dir / "metadata_sunburst"),
                                #   subfig_group="subject", 
                                  groups=["pulse_type", "task", "laser_state"],
                                  title=project.trialgroup.group_name)

In [ ]:
project.trialgroup.trial_success_rate(u.make_path(analysis_dir, "trial_success_rate.png"))

## Load trials

In [ ]:
project.trialgroup.load_trials()
bodypart = "finger_3"

In [ ]:
fig, ax = plt.subplots()
n=0

for trial in tqdm(project.trialgroup.trials, desc="plotting timeseries"): 
    if not trial.is_valid() or trial.time_pad_off is None: 
        continue

    n+=1
    tracking = DLCTracking(path=trial.pred_path,
                        view=trial.camera_view, 
                        bodyparts=project.project_info["bodyparts"],
                        time_pad_off=trial.time_pad_off,
                        cm_per_pixel=trial.cm_per_pixel,
                        shift=trial.camera_shift, 
                        status=trial.trajectories)

    ax = traj.plot_traj(traj.coords, ax, fig, color = "gray")

    ax.set(
        xlabel="X position (cm)",
        ylabel="Y position (cm)",
    )

ax.set_title(f"bp: {bodypart}, n trials: {}/{len(project.trialgroup.trials)}")
date = datetime.now().strftime("%Y%m%d")
fig.savefig(u.make_path(analysis_dir, f"{bodypart}_all_traj_({date}).svg"))
plt.close()